## 亩产量（预期销售量）

In [1]:
import pandas as pd
# 读取附件2中2023年的农作物种植情况
data2_1 = pd.read_excel('../data/附件2.xlsx', sheet_name='2023年的农作物种植情况')
data2_1['种植地块'] = data2_1['种植地块'].fillna(method='ffill')

In [2]:
# 去除data2_1中所有object类型数据的空格
for col in data2_1.select_dtypes(include=['object']).columns:
    data2_1[col] = data2_1[col].str.strip()

In [3]:
# 为data2_1新增一列，名为地块类型
def get_land_type(land):
    if land.startswith('A'):
        return '平旱地'
    elif land.startswith('B'):
        return '梯田'
    elif land.startswith('C'):
        return '山坡地'
    elif land.startswith('D'):
        return '水浇地'
    elif land.startswith('E'):
        return '普通大棚'
    elif land.startswith('F'):
        return '智慧大棚'
    else:
        return '未知'

data2_1['地块类型'] = data2_1['种植地块'].apply(get_land_type)

In [4]:
data2_1

,种植地块,作物编号,作物名称,作物类型,种植面积/亩,种植季次,地块类型
0,A1,6,小麦,粮食,80.0,单季,平旱地
1,A2,7,玉米,粮食,55.0,单季,平旱地
2,A3,7,玉米,粮食,35.0,单季,平旱地
3,A4,1,黄豆,粮食（豆类）,72.0,单季,平旱地
4,A5,4,绿豆,粮食（豆类）,68.0,单季,平旱地
...,...,...,...,...,...,...,...
82,F3,28,小青菜,蔬菜,0.3,第二季,智慧大棚
83,F3,30,生菜,蔬菜,0.3,第二季,智慧大棚
84,F4,19,芸豆,蔬菜（豆类）,0.6,第一季,智慧大棚
85,F4,34,芹菜,蔬菜,0.3,第二季,智慧大棚


In [5]:
# 读取附件2中 2023年统计的相关数据
data2_2 = pd.read_excel('../data/附件2.xlsx', sheet_name='2023年统计的相关数据').iloc[:107, :]
# 删除data2_2中序号这一列
data2_2 = data2_2.drop(columns=['序号'])


In [6]:
# 将data2_2中销售单价/(元/斤)这一列的值形如2.50-4.00的值分开，另外增加两列，分别为最小值和最大值
data2_2[['最小值', '最大值']] = data2_2['销售单价/(元/斤)'].str.split('-', expand=True).astype(float)
# 取最小值和最大值的平均值，增加一列为平均值
data2_2['平均值'] = data2_2[['最小值', '最大值']].mean(axis=1)


In [7]:
# 去除data2_2中所有object类型数据的空格
for col in data2_2.select_dtypes(include=['object']).columns:
    data2_2[col] = data2_2[col].astype(str).str.strip()

In [8]:
# 复制普通大棚第一季的数据，新增智慧大棚第一季的数据
data2_2_copy = data2_2[(data2_2['地块类型'] == '普通大棚') & (data2_2['种植季次'] == '第一季')].copy()
data2_2_copy['地块类型'] = '智慧大棚'
# 将复制的数据添加到data2_2中
data2_2 = pd.concat([data2_2, data2_2_copy], ignore_index=True)

In [9]:
data2_2["作物编号"] = data2_2["作物编号"].astype(int)

In [10]:
data2_2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,种植成本/(元/亩),销售单价/(元/斤),最小值,最大值,平均值
0,1,黄豆,平旱地,单季,400,400,2.50-4.00,2.5,4.0,3.25
1,2,黑豆,平旱地,单季,500,400,6.50-8.50,6.5,8.5,7.50
2,3,红豆,平旱地,单季,400,350,7.50-9.00,7.5,9.0,8.25
3,4,绿豆,平旱地,单季,350,350,6.00-8.00,6.0,8.0,7.00
4,5,爬豆,平旱地,单季,415,350,6.00-7.50,6.0,7.5,6.75
...,...,...,...,...,...,...,...,...,...,...
120,30,生菜,智慧大棚,第一季,5000,2000,4.50-6.00,4.5,6.0,5.25
121,31,辣椒,智慧大棚,第一季,2000,1200,6.00-8.50,6.0,8.5,7.25
122,32,空心菜,智慧大棚,第一季,12000,5000,3.00-6.00,3.0,6.0,4.50
123,33,黄心菜,智慧大棚,第一季,6000,2500,4.00-5.00,4.0,5.0,4.50


In [11]:
# 确保列名没有空格
data2_1.columns = data2_1.columns.str.strip()
data2_2.columns = data2_2.columns.str.strip()

# 将处理好的data2_2与data2_1进行内连接，连接键为作物编号和地块类型
merged_data = pd.merge(data2_2, data2_1, on=['作物编号', '地块类型','作物名称','种植季次'], how='left')
merged_data

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,种植成本/(元/亩),销售单价/(元/斤),最小值,最大值,平均值,种植地块,作物类型,种植面积/亩
0,1,黄豆,平旱地,单季,400,400,2.50-4.00,2.5,4.0,3.25,A4,粮食（豆类）,72.0
1,2,黑豆,平旱地,单季,500,400,6.50-8.50,6.5,8.5,7.50,NaN,NaN,NaN
2,3,红豆,平旱地,单季,400,350,7.50-9.00,7.5,9.0,8.25,NaN,NaN,NaN
3,4,绿豆,平旱地,单季,350,350,6.00-8.00,6.0,8.0,7.00,A5,粮食（豆类）,68.0
4,5,爬豆,平旱地,单季,415,350,6.00-7.50,6.0,7.5,6.75,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,30,生菜,智慧大棚,第一季,5000,2000,4.50-6.00,4.5,6.0,5.25,NaN,NaN,NaN
144,31,辣椒,智慧大棚,第一季,2000,1200,6.00-8.50,6.0,8.5,7.25,NaN,NaN,NaN
145,32,空心菜,智慧大棚,第一季,12000,5000,3.00-6.00,3.0,6.0,4.50,F1,蔬菜,0.3
146,33,黄心菜,智慧大棚,第一季,6000,2500,4.00-5.00,4.0,5.0,4.50,F1,蔬菜,0.3


In [12]:
merged_data = merged_data.groupby(['作物编号', '地块类型', '种植季次', '作物名称'], as_index=False).agg({
    '亩产量/斤': 'first',
    '种植成本/(元/亩)': 'first',
    '销售单价/(元/斤)': 'first',
    '最小值': 'first',
    '最大值': 'first',
    '平均值': 'first',
    '种植地块': 'first',  # 保留任意一个“种植地块”
    '种植面积/亩': 'sum'   # 将“种植面积/亩”相加
})

In [13]:
merged_data = merged_data.drop(columns=['种植地块'])

In [14]:
merged_data["预期销售量/斤"] = merged_data["亩产量/斤"] * merged_data["种植面积/亩"]


In [15]:
merged_data

,作物编号,地块类型,种植季次,作物名称,亩产量/斤,种植成本/(元/亩),销售单价/(元/斤),最小值,最大值,平均值,种植面积/亩,预期销售量/斤
0,1,山坡地,单季,黄豆,360,400,2.50-4.00,2.5,4.0,3.25,15.0,5400.0
1,1,平旱地,单季,黄豆,400,400,2.50-4.00,2.5,4.0,3.25,72.0,28800.0
2,1,梯田,单季,黄豆,380,400,2.50-4.00,2.5,4.0,3.25,60.0,22800.0
3,2,山坡地,单季,黑豆,450,400,6.50-8.50,6.5,8.5,7.50,0.0,0.0
4,2,平旱地,单季,黑豆,500,400,6.50-8.50,6.5,8.5,7.50,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
120,37,水浇地,第二季,红萝卜,3000,500,2.50-4.00,2.5,4.0,3.25,12.0,36000.0
121,38,普通大棚,第二季,榆黄菇,5000,3000,50.00-65.00,50.0,65.0,57.50,1.8,9000.0
122,39,普通大棚,第二季,香菇,4000,2000,18.00-20.00,18.0,20.0,19.00,1.8,7200.0
123,40,普通大棚,第二季,白灵菇,10000,10000,14.00-18.00,14.0,18.0,16.00,1.8,18000.0


In [16]:
# 删除种植面积/亩为0的行
merged_data2 = merged_data[merged_data['种植面积/亩'] != 0]
merged_data2

,作物编号,地块类型,种植季次,作物名称,亩产量/斤,种植成本/(元/亩),销售单价/(元/斤),最小值,最大值,平均值,种植面积/亩,预期销售量/斤
0,1,山坡地,单季,黄豆,360,400,2.50-4.00,2.5,4.0,3.25,15.0,5400.0
1,1,平旱地,单季,黄豆,400,400,2.50-4.00,2.5,4.0,3.25,72.0,28800.0
2,1,梯田,单季,黄豆,380,400,2.50-4.00,2.5,4.0,3.25,60.0,22800.0
5,2,梯田,单季,黑豆,475,400,6.50-8.50,6.5,8.5,7.50,46.0,21850.0
6,3,山坡地,单季,红豆,360,350,7.50-9.00,7.5,9.0,8.25,20.0,7200.0
...,...,...,...,...,...,...,...,...,...,...,...,...
120,37,水浇地,第二季,红萝卜,3000,500,2.50-4.00,2.5,4.0,3.25,12.0,36000.0
121,38,普通大棚,第二季,榆黄菇,5000,3000,50.00-65.00,50.0,65.0,57.50,1.8,9000.0
122,39,普通大棚,第二季,香菇,4000,2000,18.00-20.00,18.0,20.0,19.00,1.8,7200.0
123,40,普通大棚,第二季,白灵菇,10000,10000,14.00-18.00,14.0,18.0,16.00,1.8,18000.0


In [17]:
merged_data2 = merged_data2.drop(columns=['种植面积/亩','销售单价/(元/斤)','最小值','最大值','亩产量/斤'])
merged_data2["种植季次"].replace('单季', '第一季',inplace=True)
merged_data2.rename(columns={'平均值': '销售单价平均值/(元/斤)'}, inplace=True)

In [18]:
merged_data2

,作物编号,地块类型,种植季次,作物名称,种植成本/(元/亩),销售单价平均值/(元/斤),预期销售量/斤
0,1,山坡地,第一季,黄豆,400,3.25,5400.0
1,1,平旱地,第一季,黄豆,400,3.25,28800.0
2,1,梯田,第一季,黄豆,400,3.25,22800.0
5,2,梯田,第一季,黑豆,400,7.50,21850.0
6,3,山坡地,第一季,红豆,350,8.25,7200.0
...,...,...,...,...,...,...,...
120,37,水浇地,第二季,红萝卜,500,3.25,36000.0
121,38,普通大棚,第二季,榆黄菇,3000,57.50,9000.0
122,39,普通大棚,第二季,香菇,2000,19.00,7200.0
123,40,普通大棚,第二季,白灵菇,10000,16.00,18000.0


In [19]:
merged_data3 = merged_data2.groupby(['作物编号', '种植季次'], as_index=False).agg({
    '作物名称': 'first',
    '种植成本/(元/亩)': 'first',
    '销售单价平均值/(元/斤)': 'first',
    '预期销售量/斤': 'sum'
})
merged_data3

,作物编号,种植季次,作物名称,种植成本/(元/亩),销售单价平均值/(元/斤),预期销售量/斤
0,1,第一季,黄豆,400,3.25,57000.0
1,2,第一季,黑豆,400,7.50,21850.0
2,3,第一季,红豆,350,8.25,22400.0
3,4,第一季,绿豆,350,7.00,33040.0
4,5,第一季,爬豆,350,6.75,9875.0
5,6,第一季,小麦,450,3.50,170840.0
6,7,第一季,玉米,500,3.00,132750.0
7,8,第一季,谷子,360,6.75,71400.0
8,9,第一季,高粱,400,6.00,30000.0
9,10,第一季,黍子,360,7.50,12500.0


In [20]:
# 调换种植季次与作物名称列的顺序
cols = list(merged_data3.columns)
season_index = cols.index('种植季次')
crop_name_index = cols.index('作物名称')
cols[season_index], cols[crop_name_index] = cols[crop_name_index], cols[season_index]
merged_data3 = merged_data3[cols]
merged_data3

,作物编号,作物名称,种植季次,种植成本/(元/亩),销售单价平均值/(元/斤),预期销售量/斤
0,1,黄豆,第一季,400,3.25,57000.0
1,2,黑豆,第一季,400,7.50,21850.0
2,3,红豆,第一季,350,8.25,22400.0
3,4,绿豆,第一季,350,7.00,33040.0
4,5,爬豆,第一季,350,6.75,9875.0
5,6,小麦,第一季,450,3.50,170840.0
6,7,玉米,第一季,500,3.00,132750.0
7,8,谷子,第一季,360,6.75,71400.0
8,9,高粱,第一季,400,6.00,30000.0
9,10,黍子,第一季,360,7.50,12500.0


In [21]:
merged_data2.to_csv('../data/cropStatistics1.csv', index=False)
merged_data3.to_csv('../data/cropStatistics2.csv', index=False)

## 建模前的数据准备

In [22]:
data2_2 = data2_2.drop(
    columns=['销售单价/(元/斤)', '最小值', '最大值'])
data2_2["种植季次"].replace('单季', '第一季', inplace=True)
data2_2.rename(columns={'平均值': '销售单价平均值/(元/斤)'}, inplace=True)

In [23]:
data2_2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,种植成本/(元/亩),销售单价平均值/(元/斤)
0,1,黄豆,平旱地,第一季,400,400,3.25
1,2,黑豆,平旱地,第一季,500,400,7.50
2,3,红豆,平旱地,第一季,400,350,8.25
3,4,绿豆,平旱地,第一季,350,350,7.00
4,5,爬豆,平旱地,第一季,415,350,6.75
...,...,...,...,...,...,...,...
120,30,生菜,智慧大棚,第一季,5000,2000,5.25
121,31,辣椒,智慧大棚,第一季,2000,1200,7.25
122,32,空心菜,智慧大棚,第一季,12000,5000,4.50
123,33,黄心菜,智慧大棚,第一季,6000,2500,4.50


In [24]:
data_use1 = pd.merge(data2_2, merged_data3, on=['作物编号', '种植季次', '作物名称','销售单价平均值/(元/斤)'], how='left')


In [25]:
data_use1


,作物编号,作物名称,地块类型,种植季次,亩产量/斤,种植成本/(元/亩)_x,销售单价平均值/(元/斤),种植成本/(元/亩)_y,预期销售量/斤
0,1,黄豆,平旱地,第一季,400,400,3.25,400.0,57000.0
1,2,黑豆,平旱地,第一季,500,400,7.50,400.0,21850.0
2,3,红豆,平旱地,第一季,400,350,8.25,350.0,22400.0
3,4,绿豆,平旱地,第一季,350,350,7.00,350.0,33040.0
4,5,爬豆,平旱地,第一季,415,350,6.75,350.0,9875.0
...,...,...,...,...,...,...,...,...,...
120,30,生菜,智慧大棚,第一季,5000,2000,5.25,2000.0,1500.0
121,31,辣椒,智慧大棚,第一季,2000,1200,7.25,1200.0,1200.0
122,32,空心菜,智慧大棚,第一季,12000,5000,4.50,5000.0,3600.0
123,33,黄心菜,智慧大棚,第一季,6000,2500,4.50,2500.0,1800.0


In [26]:
# 删除data_use1中的种植成本/(元/亩)_x
data_use1 = data_use1.drop(columns=['种植成本/(元/亩)_x'])
# 将种植成本/(元/亩)_y名称改为种植成本/(元/亩)
data_use1.rename(columns={'种植成本/(元/亩)_y': '种植成本/(元/亩)'}, inplace=True)

In [27]:
# 查看存在缺失值的行
missing_data_rows = data_use1[data_use1.isnull().any(axis=1)]
missing_data_rows

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤
52,23,菠菜,水浇地,第一季,2700,5.75,NaN,NaN
63,34,芹菜,水浇地,第一季,5500,4.00,NaN,NaN
70,23,菠菜,普通大棚,第一季,3300,5.75,NaN,NaN
81,34,芹菜,普通大棚,第一季,6600,4.00,NaN,NaN
89,17,豇豆,智慧大棚,第二季,3200,9.60,NaN,NaN
90,18,刀豆,智慧大棚,第二季,2200,8.10,NaN,NaN
91,19,芸豆,智慧大棚,第二季,3200,7.80,NaN,NaN
92,20,土豆,智慧大棚,第二季,2200,4.50,NaN,NaN
97,25,菜花,智慧大棚,第二季,3600,6.60,NaN,NaN
98,26,包菜,智慧大棚,第二季,4100,7.80,NaN,NaN


In [28]:
# 删除data_use1中存在缺失值的行
data_use1 = data_use1.dropna().reset_index(drop=True)


In [29]:
data_use1

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0
1,2,黑豆,平旱地,第一季,500,7.50,400.0,21850.0
2,3,红豆,平旱地,第一季,400,8.25,350.0,22400.0
3,4,绿豆,平旱地,第一季,350,7.00,350.0,33040.0
4,5,爬豆,平旱地,第一季,415,6.75,350.0,9875.0
...,...,...,...,...,...,...,...,...
104,29,黄瓜,智慧大棚,第一季,15000,7.00,3500.0,9000.0
105,30,生菜,智慧大棚,第一季,5000,5.25,2000.0,1500.0
106,31,辣椒,智慧大棚,第一季,2000,7.25,1200.0,1200.0
107,32,空心菜,智慧大棚,第一季,12000,4.50,5000.0,3600.0


In [30]:
## 读取附件1中乡村的现有耕地
data1_1 = pd.read_excel('../data/附件1.xlsx', sheet_name='乡村的现有耕地').iloc[:, :3]


In [31]:
# 去除空格
data1_1["地块类型"] = data1_1["地块类型"].str.strip()
data_use1["地块类型"] = data_use1["地块类型"].str.strip()

# 检查唯一值
print(data1_1["地块类型"].unique())
print(data_use1["地块类型"].unique())

['平旱地' '梯田' '山坡地' '水浇地' '普通大棚' '智慧大棚']
['平旱地' '梯田' '山坡地' '水浇地' '普通大棚' '智慧大棚']


In [32]:
data_use2 = pd.merge(data_use1, data1_1, on=['地块类型'], how='left')
data_use2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A1,80.0
1,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A2,55.0
2,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A3,35.0
3,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A4,72.0
4,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A5,68.0
...,...,...,...,...,...,...,...,...,...,...
961,32,空心菜,智慧大棚,第一季,12000,4.50,5000.0,3600.0,F4,0.6
962,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F1,0.6
963,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F2,0.6
964,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F3,0.6


In [33]:
# 新增“豆类”列，根据作物名称确定，若作物名称中含“豆”，则豆类值为1，否则为0
data_use2['豆类'] = data_use2['作物名称'].apply(lambda x: 1 if '豆' in x else 0)

In [34]:
data_use2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A1,80.0,1
1,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A2,55.0,1
2,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A3,35.0,1
3,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A4,72.0,1
4,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A5,68.0,1
...,...,...,...,...,...,...,...,...,...,...,...
961,32,空心菜,智慧大棚,第一季,12000,4.50,5000.0,3600.0,F4,0.6,0
962,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F1,0.6,0
963,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F2,0.6,0
964,33,黄心菜,智慧大棚,第一季,6000,4.50,2500.0,1800.0,F3,0.6,0


In [35]:
# 去除data_use2中所有object类型数据的空格
for col in data_use2.select_dtypes(include=['object']).columns:
    data_use2[col] = data_use2[col].str.strip()

In [36]:
# 按照作物编号和种植季次排序
data_use2 = data_use2.sort_values(by=['作物编号', '种植季次']).reset_index(drop=True)


In [37]:
data_use2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A1,80.0,1
1,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A2,55.0,1
2,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A3,35.0,1
3,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A4,72.0,1
4,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A5,68.0,1
...,...,...,...,...,...,...,...,...,...,...,...
961,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E12,0.6,0
962,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E13,0.6,0
963,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E14,0.6,0
964,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E15,0.6,0


In [38]:
# 查看存在缺失值的行
missing_data_rows = data_use2[data_use2.isnull().any(axis=1)]
missing_data_rows

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类


In [39]:
data_use2.to_csv('../data/data_use.csv', index=False)


## 2023年先验知识

In [40]:
data2_1

,种植地块,作物编号,作物名称,作物类型,种植面积/亩,种植季次,地块类型
0,A1,6,小麦,粮食,80.0,单季,平旱地
1,A2,7,玉米,粮食,55.0,单季,平旱地
2,A3,7,玉米,粮食,35.0,单季,平旱地
3,A4,1,黄豆,粮食（豆类）,72.0,单季,平旱地
4,A5,4,绿豆,粮食（豆类）,68.0,单季,平旱地
...,...,...,...,...,...,...,...
82,F3,28,小青菜,蔬菜,0.3,第二季,智慧大棚
83,F3,30,生菜,蔬菜,0.3,第二季,智慧大棚
84,F4,19,芸豆,蔬菜（豆类）,0.6,第一季,智慧大棚
85,F4,34,芹菜,蔬菜,0.3,第二季,智慧大棚


In [41]:
# 删除作物类型列
data2_1 = data2_1.drop(columns=['作物类型'])
# 将种植季次中单季改为第一季
data2_1['种植季次'] = data2_1['种植季次'].replace('单季', '第一季')
# 去除data2_1中所有object类型数据的空格
for col in data2_1.select_dtypes(include=['object']).columns:
    data2_1[col] = data2_1[col].str.strip()

In [42]:
data2_1

,种植地块,作物编号,作物名称,种植面积/亩,种植季次,地块类型
0,A1,6,小麦,80.0,第一季,平旱地
1,A2,7,玉米,55.0,第一季,平旱地
2,A3,7,玉米,35.0,第一季,平旱地
3,A4,1,黄豆,72.0,第一季,平旱地
4,A5,4,绿豆,68.0,第一季,平旱地
...,...,...,...,...,...,...
82,F3,28,小青菜,0.3,第二季,智慧大棚
83,F3,30,生菜,0.3,第二季,智慧大棚
84,F4,19,芸豆,0.6,第一季,智慧大棚
85,F4,34,芹菜,0.3,第二季,智慧大棚


In [43]:
# 将种植地块列改为地块名称
data2_1 = data2_1.rename(columns={'种植地块': '地块名称'})

In [44]:
data2_1.to_csv('../data/crop_2023.csv', index=False)

In [45]:
result_2023 = pd.merge(data_use2, data2_1, on=[
                       '作物编号', '种植季次', '地块类型', '地块名称','作物名称'], how='left')

In [46]:
result_2023.fillna(0, inplace=True)
result_2023

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类,种植面积/亩
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A1,80.0,1,0.0
1,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A2,55.0,1,0.0
2,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A3,35.0,1,0.0
3,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A4,72.0,1,72.0
4,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A5,68.0,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
961,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E12,0.6,0,0.6
962,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E13,0.6,0,0.6
963,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E14,0.6,0,0.6
964,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E15,0.6,0,0.6


In [47]:
# 新增是否种植列
result_2023['种植'] = (result_2023['种植面积/亩'] != 0).astype(int)
result_2023

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类,种植面积/亩,种植
0,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A1,80.0,1,0.0,0
1,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A2,55.0,1,0.0,0
2,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A3,35.0,1,0.0,0
3,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A4,72.0,1,72.0,1
4,1,黄豆,平旱地,第一季,400,3.25,400.0,57000.0,A5,68.0,1,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E12,0.6,0,0.6,1
962,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E13,0.6,0,0.6,1
963,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E14,0.6,0,0.6,1
964,41,羊肚菌,普通大棚,第二季,1000,100.00,10000.0,4200.0,E15,0.6,0,0.6,1


In [48]:
import numpy as np
# 提取种植面积和是否种植列，并转换为一维数组
prophet = np.concatenate([result_2023['种植面积/亩'].values, result_2023['种植'].values])
np.save('../data/prophet.npy', prophet)

## 评分合并

In [51]:
import pandas as pd

# 读取评分表
score_result = pd.read_csv('../data/score_result.csv')
# 读取data_use
data_use = pd.read_csv('../data/data_use.csv')
data_use2 = data_use.copy()

In [52]:
data_use2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类
0,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A1,80.0,1
1,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A2,55.0,1
2,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A3,35.0,1
3,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A4,72.0,1
4,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A5,68.0,1
...,...,...,...,...,...,...,...,...,...,...,...
961,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E12,0.6,0
962,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E13,0.6,0
963,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E14,0.6,0
964,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E15,0.6,0


In [53]:
# 新增方案列
data_use2['方案'] = range(1, 967)
# 将data_use2中的方案列与score_result中的方案列进行合并
data_use2 = pd.merge(data_use2, score_result, on='方案', how='left')
# 删除data_use2中的方案列
data_use2 = data_use2.drop(columns=['方案'])
data_use2

,作物编号,作物名称,地块类型,种植季次,亩产量/斤,销售单价平均值/(元/斤),种植成本/(元/亩),预期销售量/斤,地块名称,地块面积/亩,豆类,评分
0,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A1,80.0,1,21.414005
1,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A2,55.0,1,21.414005
2,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A3,35.0,1,21.414005
3,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A4,72.0,1,21.414005
4,1,黄豆,平旱地,第一季,400.0,3.25,400.0,57000.0,A5,68.0,1,21.414005
...,...,...,...,...,...,...,...,...,...,...,...,...
961,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E12,0.6,0,100.000000
962,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E13,0.6,0,100.000000
963,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E14,0.6,0,100.000000
964,41,羊肚菌,普通大棚,第二季,1000.0,100.00,10000.0,4200.0,E15,0.6,0,100.000000


In [54]:
data_use2.to_csv('../data/data_use2.csv', index=False)